# PYNQ AXI DMA — Basic Pipeline on the Kria KR260

This notebook demonstrates how to use the **AXI Direct Memory Access (DMA)** IP core
from Python using the [PYNQ](http://www.pynq.io/) framework on the **AMD Kria KR260**
Robotics Starter Kit.

## What is AXI DMA?

The AXI DMA engine lets the Processing System (PS, i.e., the ARM cores) move data
directly to/from PL (Programmable Logic) accelerators **without CPU involvement**,
using dedicated DMA channels:

| Channel | Direction | Description |
|---------|-----------|-------------|
| **Send (MM2S)** | PS → PL | Pushes data from DDR memory into the PL fabric |
| **Receive (S2MM)** | PL → PS | Pulls results from the PL fabric back into DDR memory |

## Pipeline Architecture

```
 ┌─────────────────────────────────────────────────────┐
 │                   Kria KR260                        │
 │  ┌──────────────┐          ┌──────────────────────┐ │
 │  │  PS (ARM)    │          │  PL (FPGA Fabric)    │ │
 │  │              │          │                      │ │
 │  │  input_buf ──┼─ DMA ───►│  AXI Stream IP       │ │
 │  │              │  (MM2S)  │  (e.g. passthrough / │ │
 │  │ output_buf ◄─┼─ DMA ───-│   filter / compute)  │ │
 │  │              │  (S2MM)  │                      │ │
 │  └──────────────┘          └──────────────────────┘ │
 │              shared DDR (PSDDR)                     │
 └─────────────────────────────────────────────────────┘
```

## Notebook Flow

1. Reset the PL and import PYNQ libraries
2. Load the bitstream overlay (`kria_dma.bit`)
3. Obtain handles for the DMA send/receive channels
4. Allocate physically-contiguous DMA buffers
5. Generate a **bi-exponential waveform** and write it into the input buffer
6. Send the data through the PL via DMA and receive the result
7. Plot and compare input vs output
8. Release buffers

---
> **Hardware requirement:** The bitfile `kria_dma.bit` (and its companion `.hwh`
> metadata file) must be present in the same directory as this notebook.

## 1. Reset the Programmable Logic

Before loading a new overlay it is good practice to reset the PL.  
`PL.reset()` tears down any previously loaded bitstream and puts all PL
peripherals into a clean state, preventing stale register values from
interfering with the new design.

In [140]:
from pynq import PL

# Reset the PL fabric to a clean state before loading a new bitstream
PL.reset()

## 2. Import Libraries

| Import | Purpose |
|--------|---------|
| `Overlay` | Loads a bitstream and parses the `.hwh` hardware description |
| `MMIO` | Low-level memory-mapped I/O access (available for manual register reads) |
| `allocate` | Allocates physically-contiguous buffers in DDR that DMA can access |
| `numpy` | Array manipulation and signal generation |
| `matplotlib` | Plotting input and output waveforms |

In [1]:
from pynq import Overlay, MMIO   # Overlay: bitstream loader; MMIO: raw register access
from pynq import allocate        # DMA-capable physically-contiguous buffer allocator
import numpy as np               # Numerical arrays and math
import matplotlib.pyplot as plt  # Waveform visualisation

## 3. Load the Bitstream Overlay

`Overlay` does three things automatically:

1. **Programs the PL** with the `.bit` file.
2. **Parses the `.hwh`** (hardware handoff) file to discover every IP core and
   memory region present in the design.
3. **Creates Python driver objects** for each known IP (e.g. `DMA`, `GPIO`,
   `AXI Interrupt Controller`, …) that are accessible as attributes.

After loading we print the available IP cores and memories to confirm the
design was loaded correctly and to understand what the bitfile exposes.

In [2]:
# Load the bitstream — this programs the FPGA and parses the hardware description (.hwh)
ol = Overlay("kria_dma.bit")

# List every IP core instantiated in the PL design
# Expected: AXI Interrupt Controller, AXI DMA, Zynq UltraScale+ PS stub
print("Accesible IP-Cores:")
print(ol.ip_dict.keys())

# List available memory regions (shared DDR between PS and PL)
# PSDDR = Processing System DDR, the main system memory
print("Accesible Memories:")
print(ol.mem_dict.keys())

Accesible IP-Cores:
dict_keys(['axi_intc_0', 'axi_dma_0', 'zynq_ultra_ps_e_0'])
Accesible Memories:
dict_keys(['PSDDR'])


## 4. Get DMA Channel Handles

The overlay exposes `axi_dma_0` as a `pynq.lib.dma.DMA` driver object.  
From it we grab the two unidirectional channels:

- **`sendchannel`** (Memory-to-Stream, MM2S): reads from DDR and drives an
  AXI4-Stream master port into the PL.
- **`recvchannel`** (Stream-to-Memory, S2MM): accepts an AXI4-Stream slave
  port from the PL and writes the result back into DDR.

Separating the handles makes it clear which direction each `.transfer()` call
operates in.

In [3]:
# Obtain the AXI DMA driver object from the overlay
dma = ol.axi_dma_0

# MM2S channel: PS → PL  (sends data from DDR into the PL accelerator)
dma_send = dma.sendchannel

# S2MM channel: PL → PS  (receives processed data from the PL back into DDR)
dma_recv = dma.recvchannel

## 5. Allocate the Input Buffer

`pynq.allocate` creates a NumPy-compatible array that lives in
**physically-contiguous DDR memory**.  Unlike ordinary NumPy arrays, this
memory is directly accessible by the DMA engine without needing a software
bounce-buffer or OS scatter-gather.

- `shape=(data_size,)` — a 1-D array of `data_size` elements.
- `dtype=np.uint32` — 32-bit unsigned integers, matching the AXI4-Stream
  data width configured in the PL design (32 bits per beat).

In [4]:
# Number of samples to transfer in a single DMA transaction
data_size = 50

# Allocate a physically-contiguous buffer for the input data (PS → PL direction)
# dtype must match the AXI-Stream data width in the PL (here: 32-bit unsigned)
input_buffer = allocate(shape=(data_size,), dtype=np.uint32)

## 6. Define the Bi-Exponential Waveform

A **bi-exponential** (double-exponential) pulse is a classic waveform used in
nuclear instrumentation, radiation detection, and neuroscience to model the
charge collection of a detector or the synaptic current of a neuron.

The mathematical form is:

$$y(t) = -A \left( e^{-t/\tau_f} - e^{-t/\tau_s} \right)$$

| Parameter | Symbol | Role |
|-----------|--------|------|
| Amplitude | $A$ | Overall pulse height |
| Fast time constant | $\tau_f$ | Controls the **rise** (small value → fast rise) |
| Slow time constant | $\tau_s$ | Controls the **fall** (large value → slow decay) |

The peak appears at $t_{peak} = \frac{\tau_f \tau_s}{\tau_s - \tau_f} \ln(\tau_s/\tau_f)$.

In [5]:
def biexp(t, A, tau_f, tau_s):
    """
    Bi-exponential pulse shape.

    Parameters
    ----------
    t     : array-like  Time axis [s]
    A     : float       Amplitude scaling factor
    tau_f : float       Fast (rise) time constant [s]  — small value → fast rise
    tau_s : float       Slow (fall) time constant [s]  — large value → slow decay

    Returns
    -------
    ndarray  Pulse values (positive peak, causal signal starting from 0)
    """
    y = A * (np.exp(-t / tau_f) - np.exp(-t / tau_s))
    return -y  # Negate so the pulse points upward

## 7. Generate the Signal and Fill the Input Buffer

Steps:
1. Build a time vector spanning **0 – 30 µs** with `data_size` points.
2. Evaluate the bi-exponential pulse with $\tau_f = 0.5\,\mu s$ and
   $\tau_s = 5\,\mu s$ (ratio 1:10, typical of a fast-rise / slow-fall detector
   pulse).
3. **Scale** the normalised float values (0 – 1) to the integer range
   $[0, 2^{12}-1]$ (12-bit ADC-like encoding, since the PL design works with
   fixed-point integers).
4. Cast and copy each sample into the DMA-mapped `input_buffer`.
5. Plot the buffer to visually verify the waveform before sending it to the PL.

In [6]:
# ── Time axis ──────────────────────────────────────────────────────────────────
t = np.linspace(0, 30e-6, data_size)  # 50 equally-spaced points from 0 to 30 µs

# ── Bi-exponential parameters ──────────────────────────────────────────────────
A     = 1.0    # Normalised amplitude (will be scaled to integer range below)
tau_f = 0.5e-6 # Fast time constant: 0.5 µs  → controls the rising edge
tau_s = 5e-6   # Slow time constant: 5.0 µs  → controls the falling tail

# Evaluate the pulse shape over the time axis
y = biexp(t, A, tau_f, tau_s)

# ── Fixed-point encoding ───────────────────────────────────────────────────────
# The PL operates on unsigned 32-bit integers. Scale the normalised float signal
# to 12-bit range [0, 4095] to simulate a 12-bit ADC output.
scale = 2**12  # = 4096  →  12-bit full-scale value

for i in range(data_size):
    # Multiply, truncate to integer, and store in the DMA-mapped buffer
    input_buffer[i] = int(y[i] * scale)

# ── Verify the input waveform ──────────────────────────────────────────────────
plt.plot(t, input_buffer)
plt.title("Input signal (bi-exponential, 12-bit scaled)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (counts)")
plt.show()

## 8. Send Data to the PL via DMA (MM2S Transfer)

`dma_send.transfer(input_buffer)` programs the **MM2S** DMA channel to read
`data_size × 4` bytes from the physical address of `input_buffer` and push them
as an AXI4-Stream packet into the PL fabric.

> **Note:** By default `transfer()` is **non-blocking** — it starts the DMA
> engine and returns immediately.  Call `dma_send.wait()` if you need to
> guarantee the transfer is complete before continuing.

In [149]:
# Start the MM2S (Memory-to-Stream) DMA transfer:
# Reads input_buffer from DDR and streams it into the PL accelerator
dma_send.transfer(input_buffer)

## 9. Allocate the Output Buffer

Allocate a second DMA-capable buffer of the **same size and type** as the input.
This buffer will hold the data returned by the PL accelerator over the
S2MM (Stream-to-Memory) channel.

> The output buffer is allocated separately so it can be checked independently
> from the input, which is useful for debugging passthrough or filter pipelines.

In [150]:
# Allocate a physically-contiguous buffer to receive the PL output (PL → PS direction)
# Shape and dtype must match the input buffer and the PL AXI-Stream data width
output_buffer = allocate(shape=(data_size,), dtype=np.uint32)

## 10. Receive Data from the PL via DMA (S2MM Transfer)

`dma_recv.transfer(output_buffer)` programs the **S2MM** DMA channel to accept
the AXI4-Stream packet coming out of the PL and write it into `output_buffer`
in DDR.

In a **loopback** or **passthrough** design (where the PL streams data straight
back) the output should be identical to the input.  In a **processing** design
(filter, FFT, etc.) the output will be the transformed data.

> Again, add `dma_recv.wait()` before reading `output_buffer` if you need a
> synchronisation point.

In [151]:
# Start the S2MM (Stream-to-Memory) DMA transfer:
# Receives the processed AXI-Stream data from the PL and writes it to output_buffer in DDR
dma_recv.transfer(output_buffer)

## 11. Plot the Output Waveform

Visualise the data received from the PL.  
Compare this plot with the input plot above:

- **Identical shapes** → the PL is a pure passthrough (loopback).
- **Modified shape** → the PL applied some processing (filter, gain, delay, …).

In [152]:
# Plot the waveform returned by the PL accelerator
plt.plot(t, output_buffer)
plt.title("Output signal received from PL")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (counts)")
plt.show()

## 12. Release DMA Buffers

`pynq.allocate` buffers are backed by **CMA (Contiguous Memory Allocator)**
regions managed by the kernel.  They are **not** automatically freed by Python's
garbage collector in all PYNQ versions.

Always `del` (or call `.freebuffer()` on) allocated buffers when you are done
to return the contiguous memory to the pool and avoid memory leaks across
repeated notebook runs.

In [153]:
# Free the physically-contiguous DMA buffers to release CMA memory back to the system
del input_buffer, output_buffer

---
## Summary

| Step | API call | Description |
|------|----------|-------------|
| Reset PL | `PL.reset()` | Clear any previous bitstream |
| Load overlay | `Overlay("kria_dma.bit")` | Program FPGA and discover IPs |
| Get DMA handle | `ol.axi_dma_0` | Python driver for the DMA core |
| Allocate buffers | `allocate(shape, dtype)` | Physically-contiguous DDR buffers |
| Send to PL | `dma_send.transfer(buf)` | MM2S: DDR → AXI-Stream → PL |
| Receive from PL | `dma_recv.transfer(buf)` | S2MM: PL → AXI-Stream → DDR |
| Release memory | `del buf` | Return CMA pages to the kernel |

### Next Steps

- Replace the passthrough PL design with a **FIR filter**, **FFT**, or custom
  HLS core and re-run this notebook to see the effect on the bi-exponential pulse.
- Add `dma_send.wait()` / `dma_recv.wait()` for explicit synchronisation.
- Increase `data_size` and benchmark the throughput of your PL pipeline.